# **Fraud Analysis - Cross-Border Remittance Book 2026**

**Author:** Vincent Mbira

**Period Covered:** 1 January - Septemher 2026 (YTD as at 24th September 2026)

**Volume:** 158,020 transactions analyzed | 14,819 customers  | $43.4Mn attempted | $35.4Mn settled

### **Objective:**
Establish what normal looks like in this book, identify fraud patterns from behavioural and network evidence, evaluate how the live rules are performing against both fraud and genuine customers, and recommend changes.

### **Data & Provenance:**
Two files:

    - `remittance_transactions_2026.csv`  —> transaction ledger, 15 fields, one row per payment attempt.
    - `fraud_labels_2026.csv`  —> confirmed bad outcomes (chargebacks, confirmed takeover, mule confirmations, customer scam claims).

**N/B:** *The data is synthetic, constructed to mirror the structure of a live remittance book. Findings demonstrate method, not industry benchmarks.*


### **Methodological Choice:** labels are quarantined
The labels are deliberately **not loaded until section 8**. Sections 2–7 are performed blind, as they would be in a real investigation where fraud is identified before confirmation arrives. The labels are then used to score the findings rather than to generate them.

This matters because the labels have three properties that make them unsuitable as ground truth on their own:

    1. **Delayed** — cases confirm weeks after the payment, so recent weeks look artificially clean. That is label immaturity, not lower fraud.
    2. **Incomplete** — not every bad payment is reported. Absence of a label is weak evidence of legitimacy.
    3. **Structurally biased** — only settled payments can ever be labelled. A blocked payment never moved money and so can never generate a chargeback. Any precision measured against these labels is biased by construction, and the analysis corrects for this rather than ignoring it.

### **Clarification on the `Status` record on the transactions:**
`status` records the system's decision, not the truth. `BLOCKED_RISK` means a rule fired, not that fraud occurred. `COMPLETED` means nothing stopped the payment, not that it was genuine. Rule behaviour is therefore evaluated as a separate question from fraud detection, and block status is never used as a fraud label.

### **Conventions:**
- **Attempted** = all rows. Used for rule behaviour and fraud attempts.
- **Settled** = `status == 'COMPLETED'`. Used for money paid out and all loss figures.
- `FAILED_TECH` and `PENDING_REVIEW` are neither successes nor risk decisions: excluded from loss figures, retained in attempt counts.
- Monthly comparisons use rates per 1,000 transactions, not counts, because the book more than doubled over the period.

### **Notebook structure:**
1. Load & data-quality checks
2. Portfolio overview, baseline & EDA
3. Feature engineering - behavioral, network, & link features)
4. Fraud pattern discovery (unlabelled)
5. Rule performance evaluation
6. Prototype risk score & back-test
7. Self-checks and Validation
8. Labelled evaluation, findings, and recommendations

## **1. Load & Data-quality Checks**


In [1]:
!pip install pandas numpy matplotlib seaborn networkx

In [2]:
# loading the libraries
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

In [3]:
# setting the style of the plots

# setting the display options for pandas and matplotlib
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

# setting the style of the plots
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False, 'axes.spines.right': False})

In [4]:
# loading the dataset

filepath = "D:\\Job Application\\NALA Money Fraud Analyst\\remittance_transactions_2026.csv"
df = pd.read_csv(filepath, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
print(df.shape)
df.head()

(158140, 15)


,txn_id,timestamp,customer_id,account_age_days,send_country,receive_country,amount_usd,payment_method,card_fingerprint,device_id,ip_country,recipient_id,three_ds,status,blocked_by_rule
0,T1000000,2026-01-01 00:01:00,C008621,1056,FR,IN,887.31,APPLE_PAY,F006054,D011522,FR,R003239,U,COMPLETED,NaN
1,T1000001,2026-01-01 00:34:00,C002823,468,de,IN,173.09,CARD,F000701,D000685,DE,R009419,Y,COMPLETED,NaN
2,T1000002,2026-01-01 00:55:00,C004419,71,GB,GH,63.59,BANK_TRANSFER,NaN,D011305,GB,R020130,NaN,COMPLETED,NaN
3,T1000003,2026-01-01 01:46:00,C003763,100,GB,PK,200.00,CARD,F002891,D006937,GB,R011190,U,COMPLETED,NaN
4,T1000004,2026-01-01 02:24:00,C002836,703,US,BD,200.00,APPLE_PAY,F003516,D011785,US,R002857,Y,COMPLETED,NaN


In [5]:
df.dtypes

txn_id                         str
timestamp           datetime64[us]
customer_id                    str
account_age_days             int64
send_country                   str
receive_country                str
amount_usd                 float64
payment_method                 str
card_fingerprint               str
device_id                      str
ip_country                     str
recipient_id                   str
three_ds                       str
status                         str
blocked_by_rule                str
dtype: object

In [6]:
# checking the data types and missing values
df.dtypes
df.isna().sum()
print('rows:', len(df), '| unique txn_id:', df.txn_id.nunique())
print('fully duplicated rows:', df.duplicated().sum())
df['timestamp'] = pd.to_datetime(df.timestamp, errors='coerce')
print('unparsed timestamps:', df.timestamp.isna().sum())
print('range:', df.timestamp.min(), '->', df.timestamp.max())
for c in ['send_country','receive_country','ip_country','payment_method','three_ds','status','blocked_by_rule']:
    print(f'\n{c} ({df[c].nunique()} distinct):'); print(df[c].value_counts(dropna=False).head(12))
df.amount_usd.describe([.01,.25,.5,.75,.95,.99])
df.account_age_days.describe()

rows: 158140 | unique txn_id: 158020
fully duplicated rows: 120
unparsed timestamps: 0
range: 2026-01-01 00:01:00 -> 2026-09-23 14:05:00

send_country (10 distinct):
send_country
GB    52380
US    46891
DE    22162
FR    21296
IE    14779
gb      224
us      158
de       98
fr       89
ie       63
Name: count, dtype: int64

receive_country (20 distinct):
receive_country
IN     24026
NG     23933
PK     23233
GH     22450
KE     22082
BD     13523
CM     11296
SN     10254
UG      4277
TZ      2592
KE        79
PK        75
Name: count, dtype: int64

ip_country (18 distinct):
ip_country
GB     51969
US     46537
DE     22314
FR     21250
IE     14942
NaN      316
GH       163
NG       154
RU        84
ID        71
IN        63
PK        58
Name: count, dtype: int64

payment_method (3 distinct):
payment_method
CARD             75079
BANK_TRANSFER    45413
APPLE_PAY        37648
Name: count, dtype: int64

three_ds (3 distinct):
three_ds
Y      91485
NaN    45413
U      16448
N       4794


count    158140.000000
mean        564.593360
std         527.588036
min          -1.000000
25%          80.000000
50%         393.000000
75%        1017.000000
max        1761.000000
Name: account_age_days, dtype: float64

**Duplicates:** There are about 120 rows that are duplicated, these will be dropped.

**Fragmented Country codes:** Some of the country codes are in lowcase and others are uppercase thus we will clean them. 

**Missing IP country:** 316 records have missing ip_country entries and this could be signals of potential account takeovers. Wll inestigate them during the analysis. 

In [7]:
# cleaning the data
clean = df.drop_duplicates().copy()
for c in ['send_country','receive_country','ip_country']:
    clean[c] = clean[c].str.strip().str.upper()
clean['ip_country'] = clean['ip_country'].fillna('UNKNOWN')


In [8]:
# reconciliation trail
print(f"raw rows        : {len(df):,}")
print(f"exact duplicates: {len(df)-len(clean):,}")
print(f"clean rows      : {len(clean):,}")
print(f"unique txn_id   : {clean.txn_id.nunique():,}")

raw rows        : 158,140
exact duplicates: 120
clean rows      : 158,020
unique txn_id   : 158,020


In [9]:
# structural null assumptions, tested not assumed
bt = clean.payment_method == 'BANK_TRANSFER'
print('\ncard payments missing fingerprint:', clean.loc[~bt,'card_fingerprint'].isna().sum())
print('bank transfers with a fingerprint:', clean.loc[bt,'card_fingerprint'].notna().sum())
assert (clean.blocked_by_rule.notna() == (clean.status=='BLOCKED_RISK')).all()


card payments missing fingerprint: 0
bank transfers with a fingerprint: 0


In [10]:
print('\nsend:', sorted(clean.send_country.unique()))
print('receive:', sorted(clean.receive_country.unique()))
print('ip:', sorted(clean.ip_country.unique()))


send: ['DE', 'FR', 'GB', 'IE', 'US']
receive: ['BD', 'CM', 'GH', 'IN', 'KE', 'NG', 'PK', 'SN', 'TZ', 'UG']
ip: ['BD', 'BR', 'CM', 'DE', 'FR', 'GB', 'GH', 'ID', 'IE', 'IN', 'KE', 'NG', 'PK', 'RU', 'SN', 'TZ', 'UG', 'UNKNOWN', 'US']


In [11]:
# impossible values
print('\nnegative account_age_days:', (clean.account_age_days < 0).sum())
print('zero or negative amounts  :', (clean.amount_usd <= 0).sum())
display(clean.amount_usd.describe([.01,.25,.5,.75,.95,.99]).round(2))
display(clean.account_age_days.describe().round(1))


negative account_age_days: 40
zero or negative amounts  : 0


count    158020.00
mean        274.93
std         283.97
min           1.01
1%           40.33
25%         132.05
50%         201.97
75%         321.03
95%         685.56
99%        1369.73
max        9000.00
Name: amount_usd, dtype: float64

count    158020.0
mean        564.6
std         527.6
min          -1.0
25%          80.0
50%         393.0
75%        1017.0
max        1761.0
Name: account_age_days, dtype: float64

In [12]:
# account age repair

clean.loc[clean.account_age_days < 0, 'account_age_days'] = np.nan
print('corrupted account ages:', clean.account_age_days.isna().sum())

# reconstruct from the customer's implied signup date
clean['signup'] = clean.timestamp.dt.normalize() - pd.to_timedelta(clean.account_age_days, unit='D')
sign = clean.groupby('customer_id').signup.median()

# consistency check: does each customer's implied signup hold still?
spread = clean.groupby('customer_id').signup.agg(lambda s: (s.max()-s.min()).days if s.notna().any() else np.nan)
print('customers with signup spread > 1 day:', (spread > 1).sum())

fix = clean.account_age_days.isna()
clean.loc[fix, 'account_age_days'] = (
    clean.loc[fix, 'timestamp'].dt.normalize() - clean.loc[fix, 'customer_id'].map(sign)
).dt.days
print('still missing after repair:', clean.account_age_days.isna().sum())
print('new minimum:', clean.account_age_days.min())

corrupted account ages: 40
customers with signup spread > 1 day: 0
still missing after repair: 2
new minimum: 0.0


## 2.**Portfolio Overview**

In [13]:
print(f"transactions : {len(clean):,}")
print(f"customers    : {clean.customer_id.nunique():,}")
print(f"recipients   : {clean.recipient_id.nunique():,}")
print(f"devices      : {clean.device_id.nunique():,}")
print(f"total sent   : ${clean.amount_usd.sum():,.0f}")
settled = clean[clean.status == 'COMPLETED']
print(f"settled value: ${settled.amount_usd.sum():,.0f}")

display(clean.amount_usd.describe([.01,.25,.5,.75,.9,.95,.99]).round(2))

# outcome mix
display(clean.status.value_counts().to_frame('n').assign(pct=lambda x: (100*x.n/len(clean)).round(2)))

# monthly trend: volume, value, and outcome rates
m = clean.groupby(clean.timestamp.dt.to_period('M')).agg(
    txns=('txn_id','size'),
    value=('amount_usd','sum'),
    avg_ticket=('amount_usd','mean'),
    customers=('customer_id','nunique'),
    decline_rate=('status', lambda s: (s=='DECLINED_ISSUER').mean()),
    block_rate=('status', lambda s: (s=='BLOCKED_RISK').mean()))
display(m.round(3))

# corridor mix
corr = clean.groupby(['send_country','receive_country']).agg(
    txns=('txn_id','size'), value=('amount_usd','sum'), median_amt=('amount_usd','median'))
display(corr.sort_values('txns', ascending=False).head(15).round(0))

transactions : 158,020
customers    : 14,819
recipients   : 22,673
devices      : 14,642
total sent   : $43,443,966
settled value: $35,426,160


count    158020.00
mean        274.93
std         283.97
min           1.01
1%           40.33
25%         132.05
50%         201.97
75%         321.03
90%         502.84
95%         685.56
99%        1369.73
max        9000.00
Name: amount_usd, dtype: float64

,n,pct
status,,
COMPLETED,139215,88.10
DECLINED_ISSUER,13315,8.43
BLOCKED_RISK,4568,2.89
FAILED_TECH,622,0.39
PENDING_REVIEW,300,0.19


,txns,value,avg_ticket,customers,decline_rate,block_rate
timestamp,,,,,,
2026-01,10288,2840831.72,276.131,4590,0.085,0.017
2026-02,10157,2804237.53,276.089,4918,0.088,0.011
2026-03,14776,4094832.30,277.127,6106,0.085,0.012
2026-04,14510,3950263.26,272.244,6502,0.086,0.011
2026-05,20236,5579452.17,275.719,7946,0.085,0.030
2026-06,19585,5327808.86,272.035,8273,0.085,0.037
2026-07,22896,6383412.51,278.800,9295,0.083,0.040
2026-08,26389,7252505.10,274.831,10468,0.082,0.037
2026-09,19183,5210622.07,271.627,9402,0.082,0.037


txns      value  median_amt
send_country receive_country                              
GB           NG               10774  2855668.0       200.0
             PK                9767  3099148.0       244.0
US           IN                9690  3374478.0       259.0
GB           GH                8382  2042628.0       189.0
             KE                8177  1898611.0       175.0
US           NG                7243  2010927.0       206.0
             KE                7090  1623712.0       176.0
GB           IN                6394  2119196.0       250.0
US           PK                6157  1937126.0       242.0
             GH                5437  1398893.0       197.0
             BD                4914  1410274.0       216.0
FR           SN                4490  1182460.0       200.0
GB           BD                3997  1111873.0       203.0
FR           CM                3893   964206.0       193.0
DE           IN                3427  1233692.0       264.0

## 3.**Feature Engineering**

## 4.**Fraud Pattern Deep-dives**

## 5.**Rule Performance Evaluation**

## 6.**Prototype Risk Score & Back-test**

## 7.**Self-checks on the Analysis**

## 8.**Summary**
- Find the key insights from the analysis and recommended improvements. 